# Llama-3-8B: prune 8 layers with ReplaceMe, heal, restore, compare with fine-tuning

Code comes from `../ReplaceMe` (imported in cell A); outputs go to `REPLACEME_RUNS` (default `/home/tnn/LLM_optimisatoin/runs_reproduce`).


| Cell | Stage | Kernel |
|---|---|---|
| 1, A, B, C, D | config and code | after every restart |
| 2 | eval base | restart after |
| 3, 4 | train SFT, eval SFT | restart after each |
| 4b, 4c | profile (`distance_scored`), top-1 block | restart after 4b |
| 5 | prune (`lstsq_joint`) | restart after |
| 5b | eval pruned, T only | restart after |
| 6, 7 | heal, eval pruned + heal | restart after each |
| 8, 9 | restore with T / without T + eval | restart after each |
| 10 | summary, McNemar, comparison with the earlier run | — |

# 1. Config

In [3]:

import os, sys, gc, json, time, shutil, copy, math, io, re, hashlib, inspect, importlib, logging
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from typing import Optional, List, Tuple
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1")
os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")
import torch, torch.nn as nn, pandas as pd
from tqdm import tqdm
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from safetensors import safe_open

# repo root (this notebook lives in <repo>/notebooks); REPLACEME_REPO overrides it
REPO = Path(os.environ.get("REPLACEME_REPO", Path.cwd().parent)).resolve()
assert (REPO / "ReplaceMe" / "utils.py").exists(), REPO
sys.path.insert(0, str(REPO))

COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
RUNS = Path(os.environ.get("REPLACEME_RUNS", "/home/tnn/LLM_optimisatoin/runs_reproduce"))   # outputs: models, adapters, results
OLD_RUN = RUNS / "llama-3-8b-turing-nf4"
REUSE_SFT_FROM_OLD = False
PROFILE_BATCH = 1                                  

@dataclass
class Cfg:
    model_path: str = "unsloth/llama-3-8b"
    model_tag: str = "llama-3-8b-repo-nf4"
    layers_to_skip: int = 8
    selected_block: Optional[Tuple[int, int]] = None
    seed: int = 42
    calib_size: int = 128; calib_max_length: int = 256; calib_batch_size: int = 2
    alpha_act: float = 1.0; alpha_grad: float = 0.5; alpha_reg: float = 0.0; identity_blend_threshold: float = 0.15
    train_batch_size: int = 2; train_grad_accum: int = 4
    train_limit: Optional[int] = None; train_max_length: int = 384
    train_learning_rate: float = 2e-4; train_epochs: float = 1.0; train_warmup_steps: int = 20
    train_gradient_checkpointing: bool = True; train_optim: str = "paged_adamw_8bit"
    lora_rank: int = 8; lora_alpha: int = 16
    lora_target_modules: List[str] = field(default_factory=lambda: ["gate_proj", "up_proj", "down_proj"])
    eval_batch_size: int = 8; eval_max_new_tokens: int = 256; eval_limit: Optional[int] = None
    eval_dataset: str = "gsm8k"
    precision: str = "nf4"
    @property
    def run_dir(self): return RUNS / self.model_tag
    @property
    def results_dir(self): return self.run_dir / "results"
    @property
    def sft_adapter_dir(self): return (OLD_RUN if REUSE_SFT_FROM_OLD else self.run_dir) / "sft_adapter"
    @property
    def heal_adapter_dir(self): return self.run_dir / "heal_adapter"
    @property
    def pruned_dir(self): return self.run_dir / "pruned_ReplaceMe_joint_lstsq_1"
    def restored_dir(self, remove_T): return self.run_dir / ("restored_noT" if remove_T else "restored")

cfg = Cfg(); cfg.results_dir.mkdir(parents=True, exist_ok=True)
BLOCK_FILE = cfg.run_dir / "selected_block.json"
if BLOCK_FILE.exists():
    cfg.selected_block = tuple(json.load(open(BLOCK_FILE))["block"])
def done(stage): return (cfg.results_dir / f"{stage}_nf4.json").exists()
def vram(tag=""): print(f"vram {tag}:", round(torch.cuda.memory_allocated() / 2**30, 2), "GiB")
print("run_dir:", cfg.run_dir, "| block:", cfg.selected_block, "| compute dtype:", COMPUTE_DTYPE, "|", torch.cuda.get_device_name(0)); vram()

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


run_dir: /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4 | block: None | compute dtype: torch.float16 | NVIDIA GeForce RTX 2080 Ti
vram : 0.0 GiB


### A. ReplaceMe modules imported

In [4]:

import ReplaceMe.utils as rm_utils
import ReplaceMe.distance_scored as ds
import ReplaceMe.lstsq_joint as lj

p = Path(ds.__file__)
text = p.read_text()
if not any(k in text for k in ("hidden_states[1:]", "hs_full[1:]", "enter, leave")):
    shutil.copy(p, p.with_name("distance_scored.py.bak_nofix"))
    text, n = re.subn(r"(\n[ \t]*hs = out\.hidden_states)[ \t]*\n",
                      r"\1[1:]   # hs[l] = output of layer l: candidate i = block (i+1, i+1+skip)\n", text, count=1)
    assert n == 1, "line 'hs = out.hidden_states' not found"
    p.write_text(text)
    print("distance_scored.py fixed, backup: distance_scored.py.bak_nofix")

ds = importlib.reload(ds)
lj = importlib.reload(lj)
def md5(path): return hashlib.md5(Path(path).read_bytes()).hexdigest()[:8]
for m in (rm_utils, ds, lj):
    print(m.__name__, m.__file__, md5(m.__file__))
ds_sig = inspect.signature(ds.profile_distances).parameters
lj_sig = inspect.signature(lj.lstsq).parameters
assert {"answer_only_loss", "csv_save_path", "distances_save_path"} <= set(ds_sig), list(ds_sig)
assert {"selected_blocks", "alpha_grad", "alpha_act", "identity_blend_threshold", "num_A"} <= set(lj_sig), list(lj_sig)
assert any(k in inspect.getsource(ds.profile_distances) for k in ("hidden_states[1:]", "hs_full[1:]", "enter, leave"))


def gsm8k_calib(dataset, dataset_subset, dataset_column, dataset_size, batch_size, tokenizer):
    assert dataset == "openai/gsm8k", dataset
    data = load_dataset("openai/gsm8k", "main", split=dataset_subset)
    texts = [f"Question: {q}\nAnswer: {a}" for q, a in zip(data["question"], data["answer"])][:dataset_size]
    print(f"calibration: {len(texts)} texts, batch {batch_size}")
    return [texts[i:i + batch_size] for i in range(0, len(texts) - batch_size + 1, batch_size)]


class Loader:
    """Stands in for AutoModelForCausalLM inside a ReplaceMe module: fixed GPU placement, frozen weights.
    Gradients still reach the activations through enable_input_require_grads."""
    def __init__(self, device_map):
        self.device_map = device_map

    def from_pretrained(self, path, *args, **kw):
        if kw.get("device_map") == "cpu" or not torch.cuda.is_available():
            return AutoModelForCausalLM.from_pretrained(path, *args, **kw)
        kw["device_map"] = self.device_map
        model = AutoModelForCausalLM.from_pretrained(path, *args, **kw)
        model.requires_grad_(False)         
        model.enable_input_require_grads()
        dmap = getattr(model, "hf_device_map", None) or {"": str(next(model.parameters()).device)}
        print("device map:", dict(Counter(str(d) for d in dmap.values())), "| vram GiB:", round(torch.cuda.memory_allocated() / 2**30, 2))
        return model


ds.get_calib_dataloader = gsm8k_calib
lj.get_calib_dataloader = gsm8k_calib

ds.AutoModelForCausalLM = Loader({"": 0})

lj.AutoModelForCausalLM = Loader({"model": 0, "lm_head": 1} if torch.cuda.device_count() > 1 else {"": 0})


def read_tensor(model_dir, name):
    model_dir = Path(model_dir)
    index = model_dir / "model.safetensors.index.json"
    files = [model_dir / json.load(open(index))["weight_map"][name]] if index.exists() else sorted(model_dir.glob("*.safetensors"))
    for f in files:
        with safe_open(str(f), "pt") as st:
            if name in st.keys():
                return st.get_tensor(name)
    raise KeyError(name)


def model_dir(path):
    path = Path(path)
    if path.is_dir(): return path
    from huggingface_hub import snapshot_download
    return Path(snapshot_download(str(path), allow_patterns=["*.json", "*.safetensors"]))

Seeded with seed: 42
Seeded with seed: 42
ReplaceMe.utils /home/tnn/LLM_optimisatoin/ReplaceMe/ReplaceMe/utils.py 2c0e5eab
ReplaceMe.distance_scored /home/tnn/LLM_optimisatoin/ReplaceMe/ReplaceMe/distance_scored.py 0e00da9c
ReplaceMe.lstsq_joint /home/tnn/LLM_optimisatoin/ReplaceMe/ReplaceMe/lstsq_joint.py 47396252


### B. ras/data.py — промпты и метрика

In [5]:
import re


PROMPT_TEMPLATE = "Question: {question}\nAnswer:"
TRAIN_TEMPLATE = "Question: {question}\nAnswer: {answer}"
LABELLED_ANSWER = re.compile(r"####\s*(-?\d+(?:,\d{3})*(?:\.\d+)?)")
ANY_NUMBER = re.compile(r"-?\d+(?:,\d{3})*(?:\.\d+)?")
CONTINUATION_MARKERS = ("\nQuestion:", "[Question]", "\nQ:", "\nProblem:")


def truncate_at_continuation(text: str) -> str:
    cut = len(text)
    for marker in CONTINUATION_MARKERS:
        index = text.find(marker)
        if index != -1:
            cut = min(cut, index)
    return text[:cut]


def load_gsm8k(split: str, limit=None):
    dataset = load_dataset("openai/gsm8k", "main", split=split)
    return dataset.select(range(limit)) if limit else dataset


def numbers_match(predicted, gold, tolerance: float = 1e-4) -> bool:
    if predicted is None or gold is None:
        return False
    try:
        x, y = float(predicted), float(gold)
    except (TypeError, ValueError):
        return predicted == gold
    return abs(x - y) <= tolerance * max(1.0, abs(y))


def extract_final_answer(text: str):
    text = truncate_at_continuation(text.strip())
    labelled = LABELLED_ANSWER.search(text)
    if labelled:
        return labelled.group(1).replace(",", "")
    numbers = ANY_NUMBER.findall(text)
    return numbers[-1].replace(",", "") if numbers else None


def build_training_texts(dataset, eos_token: str):
    return [TRAIN_TEMPLATE.format(question=q, answer=a) + eos_token
            for q, a in zip(dataset["question"], dataset["answer"])]


def build_prompts(dataset):
    return [PROMPT_TEMPLATE.format(question=q) for q in dataset["question"]]

### C. ras/evaluate.py — оценка GSM8K батчами

In [6]:

def quantization_config(precision):
    if precision == "bf16": return None
    if precision == "int8": return BitsAndBytesConfig(load_in_8bit=True)
    if precision == "nf4": return BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                                      bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=COMPUTE_DTYPE)
    raise ValueError(precision)
def is_quantized(precision): return precision != "bf16"

def load_for_generation(model_path: str, adapter_path: Optional[str] = None,
                        precision: str = "nf4"):
    tokenizer = AutoTokenizer.from_pretrained(adapter_path or model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model = AutoModelForCausalLM.from_pretrained(
        model_path, dtype=COMPUTE_DTYPE, device_map={"": 0},
        quantization_config=quantization_config(precision))
    if adapter_path is not None:
        model = PeftModel.from_pretrained(model, adapter_path)
        if not is_quantized(precision):
            model = model.merge_and_unload()

    model.eval()
    model.config.use_cache = True
    model.generation_config.do_sample = False
    for attribute in ("temperature", "top_p", "top_k", "typical_p"):
        if hasattr(model.generation_config, attribute):
            setattr(model.generation_config, attribute, None)
    return model, tokenizer


def evaluate_gsm8k(model, tokenizer, batch_size: int, max_new_tokens: int,
                   limit: Optional[int] = None,
                   dataset_name: str = "gsm8k") -> dict:
    if dataset_name == "gsm8k":
        dataset = load_gsm8k("test", limit)
        prompts = build_prompts(dataset)
        gold = [extract_final_answer(answer) for answer in dataset["answer"]]
    else:
        raise ValueError(f"unknown eval_dataset: {dataset_name!r}")

    prompt_lengths = [len(tokenizer(p)["input_ids"]) for p in prompts]
    order = sorted(range(len(prompts)), key=lambda i: prompt_lengths[i])

    predictions = [None] * len(prompts)
    generations = [None] * len(prompts)
    started = time.perf_counter()

    for start in tqdm(range(0, len(order), batch_size), desc=f"Evaluating {dataset_name}"):
        indices = order[start:start + batch_size]
        encoded = tokenizer([prompts[i] for i in indices],
                            return_tensors="pt", padding=True)
        encoded = {k: v.to(model.device) for k, v in encoded.items()}

        with torch.inference_mode():
            outputs = model.generate(**encoded, max_new_tokens=max_new_tokens,
                                     do_sample=False,
                                     pad_token_id=tokenizer.eos_token_id,
                                     eos_token_id=tokenizer.eos_token_id)

        completions = tokenizer.batch_decode(
            outputs[:, encoded["input_ids"].shape[1]:], skip_special_tokens=True)
        for index, completion in zip(indices, completions):
            generations[index] = completion
            predictions[index] = extract_final_answer(completion)

    correct = sum(numbers_match(p, g) for p, g in zip(predictions, gold))
    elapsed = time.perf_counter() - started

    return {
        "total": len(prompts),
        "correct": correct,
        "exact_match": correct / len(prompts),
        "seconds": round(elapsed, 1),
        "batch_size": batch_size,
        "max_new_tokens": max_new_tokens,
        "predictions": [
            {"id": i, "predicted": predictions[i], "gold": gold[i],
             "correct": numbers_match(predictions[i], gold[i]),
             "generation": generations[i]}
            for i in range(len(prompts))
        ],
    }


def run_evaluation(config, stage: str, model_path: str,
                   adapter_path: Optional[str] = None, suffix_tag: str = "") -> dict:
    model, tokenizer = load_for_generation(model_path, adapter_path,
                                           config.precision)

    result = evaluate_gsm8k(model, tokenizer, config.eval_batch_size,
                            config.eval_max_new_tokens, config.eval_limit,
                            config.eval_dataset)
    result["stage"] = stage
    result["model_path"] = str(model_path)
    result["adapter_path"] = str(adapter_path) if adapter_path else None
    result["layers"] = model.config.num_hidden_layers
    result["precision"] = config.precision
    result["eval_dataset"] = config.eval_dataset

    suffix = suffix_tag + f"_{config.precision}"
    output_path = Path(config.results_dir) / f"{stage}{suffix}.json"
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"{stage}: EM {result['exact_match'] * 100:.2f}% "
          f"({result['correct']}/{result['total']}) "
          f"on {result['layers']} layers in {result['seconds'] / 60:.1f} min "
          f"-> {output_path}")

    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result

### D. ras/train.py (LoRA r8 на MLP, 1 эпоха)

In [7]:
IGNORE_INDEX = -100


class CausalDataset(torch.utils.data.Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.labels = labels

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return {"input_ids": self.input_ids[index],
                "attention_mask": self.attention_mask[index],
                "labels": self.labels[index]}


def _gsm8k_dataset(tokenizer, max_length: int, limit=None):
    dataset = load_gsm8k("train", limit)
    texts = build_training_texts(dataset, tokenizer.eos_token)
    encodings = tokenizer(texts, truncation=True, max_length=max_length,
                          padding=False)
    return CausalDataset(encodings["input_ids"], encodings["attention_mask"],
                         [list(ids) for ids in encodings["input_ids"]])


def train_lora(config, model_path: str, adapter_dir: Path,
               answer_only: bool = False,
               epochs: Optional[float] = None) -> Path:
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    train_dataset = _gsm8k_dataset(tokenizer, config.train_max_length, config.train_limit)

    model = AutoModelForCausalLM.from_pretrained(
        model_path, dtype=COMPUTE_DTYPE, device_map={"": 0},
        quantization_config=quantization_config(config.precision))
    model.config.use_cache = False
    if is_quantized(config.precision):
        model = prepare_model_for_kbit_training(
            model,
            use_gradient_checkpointing=config.train_gradient_checkpointing)

    model = get_peft_model(model, LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=config.lora_target_modules,
    ))
    model.print_trainable_parameters()

    adapter_dir = Path(adapter_dir)
    adapter_dir.mkdir(parents=True, exist_ok=True)

    model.is_parallelizable = True
    model.model_parallel = True
    targs = TrainingArguments(
            output_dir=str(adapter_dir / "checkpoints"),
            per_device_train_batch_size=config.train_batch_size,
            gradient_accumulation_steps=config.train_grad_accum,
            num_train_epochs=epochs if epochs is not None else config.train_epochs,
            learning_rate=config.train_learning_rate,
            warmup_steps=config.train_warmup_steps,
            lr_scheduler_type="linear",
            weight_decay=0.01,
            optim=config.train_optim,
            gradient_checkpointing=config.train_gradient_checkpointing,
            bf16=COMPUTE_DTYPE == torch.bfloat16,
            fp16=COMPUTE_DTYPE == torch.float16,
            logging_steps=10,
            save_strategy="no",
            report_to="none",
            seed=config.seed,
            remove_unused_columns=False,
            dataloader_pin_memory=False,
        )
    _ = targs.device
    targs._n_gpu = 1                     
    assert targs.n_gpu == 1

    trainer = Trainer(
        model=model,
        args=targs,
        train_dataset=train_dataset,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model,
                                             padding=True,
                                             label_pad_token_id=IGNORE_INDEX),
    )

    trainer.train()
    model.save_pretrained(str(adapter_dir))
    tokenizer.save_pretrained(str(adapter_dir))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return adapter_dir


def get_decoder_layers(model) -> nn.ModuleList:
    if hasattr(model, "transformer"):
        return model.transformer.h
    return model.model.layers


def set_decoder_layers(model, layers: nn.ModuleList) -> None:
    if hasattr(model, "transformer"):
        model.transformer.h = layers
    else:
        model.model.layers = layers


def restore_removed_layers(base_model_path: str, pruned_dir: str,
                           adapter_dir: str, output_dir: str,
                           block: Tuple[int, int], remove_transform: bool = False) -> Path:
    start, end = block
    carrier = start - 1

    pruned = AutoModelForCausalLM.from_pretrained(
        pruned_dir, dtype=torch.bfloat16, device_map="cpu")
    w_t = get_decoder_layers(pruned)[carrier].mlp.down_proj.weight.detach().clone()
    healed = PeftModel.from_pretrained(pruned, adapter_dir).merge_and_unload()

    base = AutoModelForCausalLM.from_pretrained(
        base_model_path, dtype=torch.bfloat16, device_map="cpu")
    base_layers = get_decoder_layers(base)

    layers = list(get_decoder_layers(healed))
    if remove_transform:
        w_orig = base_layers[carrier].mlp.down_proj.weight
        delta = layers[carrier].mlp.down_proj.weight.data - w_t          # ≈ B·A
        layers[carrier].mlp.down_proj.weight.data = (w_orig + delta).to(w_t.dtype)
        print(f"restore: T removed from layer {carrier}; |T·W-W|/|W|={((w_t - w_orig).float().norm() / w_orig.float().norm()):.3f}, |LoRA|/|W|={(delta.float().norm() / w_orig.float().norm()):.4f}")
    for index in range(start, end):
        layers.insert(index, copy.deepcopy(base_layers[index]))

    set_decoder_layers(healed, nn.ModuleList(layers))
    healed.config.num_hidden_layers = len(layers)
    base_layer_types = getattr(base.config, "layer_types", None)
    if base_layer_types is not None:
        healed.config.layer_types = list(base_layer_types)

    if len(layers) != base.config.num_hidden_layers:
        raise RuntimeError(
            f"restored {len(layers)} layers, base has {base.config.num_hidden_layers}")

    output_dir = Path(output_dir)
    healed.save_pretrained(str(output_dir), safe_serialization=True)
    AutoTokenizer.from_pretrained(adapter_dir).save_pretrained(str(output_dir))

    del healed, base, pruned
    gc.collect()
    torch.cuda.empty_cache()
    return output_dir

### 2. E0: Baseline model

In [6]:

if done("baseline"): print("есть")
else: run_evaluation(cfg, "baseline", cfg.model_path)
gc.collect(); torch.cuda.empty_cache()

2026-10-06 08:07:06 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:07:06 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-06 08:07:06 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 08:07:07 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:07:07 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-06 08:07:07 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/tree/main/additional_chat_templates?recursive=false&

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 08:07:17 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:07:17 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"
2026-10-06 08:07:17 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:07:17 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 08:07:17 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 08:07:18 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP

baseline: EM 10.77% (142/1319) on 32 layers in 35.0 min -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/results/baseline_nf4.json


### 3. E1: fine-tuned

In [7]:

if (cfg.sft_adapter_dir / "adapter_config.json").exists(): print("есть:", cfg.sft_adapter_dir)
else:
    vram("before"); train_lora(cfg, cfg.model_path, cfg.sft_adapter_dir)
gc.collect(); torch.cuda.empty_cache()

vram before: 0.01 GiB


2026-10-06 08:42:21 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:42:21 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 08:42:22 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:42:22 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-06 08:42:22 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-06 08:42:22 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/tree/main?recursive=true&expand=

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 08:42:33 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 08:42:33 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"


trainable params: 14,155,776 || all params: 8,044,417,024 || trainable%: 0.1760


Step,Training Loss
10,1.665943
20,1.294955
30,1.121109
40,1.025936
50,0.988918
60,1.021217
70,0.968970
80,0.963150
90,1.001908
100,0.962194


2026-10-06 09:29:34 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:34 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 09:29:34 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:34 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"


### 4. E1: evaluation

In [8]:
if done("sft"): print("есть")
else: run_evaluation(cfg, "sft", cfg.model_path, str(cfg.sft_adapter_dir))
gc.collect(); torch.cuda.empty_cache()

2026-10-06 09:29:37 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:37 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 09:29:38 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:38 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 09:29:45 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:45 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"
2026-10-06 09:29:46 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 09:29:46 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 09:29:46 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 09:29:47 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP

sft: EM 59.59% (786/1319) on 32 layers in 40.9 min -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/results/sft_nf4.json


### 4b. Profile with ReplaceMe/distance_scored.py

In [8]:
csv_path = cfg.run_dir / f"candidates_skip{cfg.layers_to_skip}.csv"
if csv_path.exists():
    print("profile on disk:", csv_path)
else:
    kw = dict(model_path=cfg.model_path, dataset="openai/gsm8k", dataset_column="text", batch_size=PROFILE_BATCH,
              max_length=cfg.calib_max_length, layers_to_skip=cfg.layers_to_skip, dataset_size=cfg.calib_size,
              dataset_subset="train", use_4bit=cfg.precision == "nf4", answer_only_loss=True, compute_taylor=False,
              taylor_veto_quantile=None, csv_save_path=str(csv_path),
              distances_save_path=str(cfg.run_dir / f"distances_skip{cfg.layers_to_skip}.pth"))
    kw = {k: v for k, v in kw.items() if k in ds_sig}
    print(kw); vram("before")
    ds.profile_distances(**kw)
gc.collect(); torch.cuda.empty_cache()
prof = pd.read_csv(csv_path)
n_layers = 32
assert len(prof) == n_layers - cfg.layers_to_skip, len(prof)
assert prof["dist_grad"].notna().all(), "no dist_grad: the gradients did not reach the hidden states"

{'model_path': 'unsloth/llama-3-8b', 'dataset': 'openai/gsm8k', 'dataset_column': 'text', 'batch_size': 1, 'max_length': 256, 'layers_to_skip': 8, 'dataset_size': 128, 'dataset_subset': 'train', 'use_4bit': True, 'answer_only_loss': True, 'compute_taylor': False, 'taylor_veto_quantile': None, 'csv_save_path': '/home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/candidates_skip8.csv', 'distances_save_path': '/home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/distances_skip8.pth'}
vram before: 0.0 GiB


2026-10-06 10:14:07 [INFO] compute dtype: torch.float16 (capability (7, 5))
2026-10-06 10:14:07 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:14:07 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-06 10:14:08 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 10:14:08 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:14:08 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"
2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/custom_generate/generate.py "HTTP/1.1 404 Not Found"


device map: {'cuda:0': 1} | vram GiB: 5.31


2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 10:14:14 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:14:15 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/tokenizer_config.json "HTTP/1.1 200 OK"
2026-10-06 10:14:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
2026-10-06 10:14:15 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/tree/main?recursive=true&expand=

calibration: 128 texts, batch 1


2026-10-06 10:14:19 [INFO] 24 candidates | signals: dist_act (all-token), dist_grad | answer_only_loss ('Answer:')
Profiling blocks: 100%|██████████| 128/128 [01:01<00:00,  2.10it/s]
2026-10-06 10:15:20 [INFO] CSV -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/candidates_skip8.csv | distances -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/distances_skip8.pth
2026-10-06 10:15:20 [INFO] top-3 by combined score:
2026-10-06 10:15:20 [INFO]   #1: block (22,30)  act=0.2950 (r0)  grad=0.3927 (r1)  score=0.5
2026-10-06 10:15:20 [INFO]   #2: block (21,29)  act=0.3005 (r2)  grad=0.3903 (r0)  score=1.0
2026-10-06 10:15:20 [INFO]   #3: block (20,28)  act=0.3000 (r1)  grad=0.3975 (r2)  score=1.5
2026-10-06 10:15:20 [INFO] SELECTED: (22, 30) (combined rank score; pre-heal signal -- for the gold standard run held-out arbitration on the top-3)


### 4c. Top-1 block: score

In [9]:

prof = pd.read_csv(cfg.run_dir / f"candidates_skip{cfg.layers_to_skip}.csv")
prof["rank_act"] = prof["dist_act"].rank(method="first") - 1
prof["rank_grad"] = prof["dist_grad"].rank(method="first") - 1
prof["score_ras"] = (prof["rank_act"] + prof["rank_grad"]) / 2
order = prof.sort_values(["score_ras", "dist_act"]).reset_index(drop=True)
block = (int(order.loc[0, "block_start"]), int(order.loc[0, "block_end"]))

json.dump({"block": block, "deleted_0based": list(range(*block)), "carrier_0based": block[0] - 1,
           "rule": "top-1 by mean rank of dist_act and dist_grad, ties by dist_act", "profile_csv": str(csv_path),
           "distance_scored": md5(ds.__file__)}, open(BLOCK_FILE, "w"), indent=2)
cfg.selected_block = block
rank_22_30 = order.index[(order.block_start == 22) & (order.block_end == 30)].tolist()
print(f"block {block}: deleted layers {list(range(*block))}, T goes into layer {block[0] - 1}")
print("earlier run used (22, 30); its rank in this profile:", rank_22_30[0] + 1 if rank_22_30 else "not in the profile")
order.head(5)[["block_start", "block_end", "dist_act", "dist_grad", "rank_act", "rank_grad", "score_ras"]]

block (22, 30): deleted layers [22, 23, 24, 25, 26, 27, 28, 29], T goes into layer 21
earlier run used (22, 30); its rank in this profile: 1


,block_start,block_end,dist_act,dist_grad,rank_act,rank_grad,score_ras
0,22,30,0.295001,0.392701,0.0,1.0,0.5
1,21,29,0.300527,0.390265,2.0,0.0,1.0
2,20,28,0.300008,0.397535,1.0,2.0,1.5
3,19,27,0.310096,0.402730,3.0,3.0,3.0
4,23,31,0.313601,0.419852,4.0,5.0,4.5


### 5. Prune with ReplaceMe/lstsq_joint.py

In [10]:

assert cfg.selected_block, "run 4c first"
buf = io.StringIO(); h = logging.StreamHandler(buf); h.setLevel(logging.WARNING); logging.getLogger().addHandler(h)
if (cfg.pruned_dir / "config.json").exists(): print("pruned model on disk:", cfg.pruned_dir)
else:
    vram("before")
    out = lj.lstsq(model_path=cfg.model_path, dataset="openai/gsm8k", dataset_column="text",
                   batch_size=cfg.calib_batch_size, max_length=cfg.calib_max_length, layers_to_skip=cfg.layers_to_skip,
                   dataset_size=cfg.calib_size, dataset_subset="train", use_4bit=cfg.precision == "nf4",
                   save_path=str(cfg.run_dir / "pruned"), alpha=cfg.alpha_reg, alpha_act=cfg.alpha_act,
                   alpha_grad=cfg.alpha_grad, identity_blend_threshold=cfg.identity_blend_threshold,
                   num_A=1, selected_blocks=[cfg.selected_block], save_transform_only=True)
    assert Path(out).resolve() == cfg.pruned_dir.resolve(), (out, cfg.pruned_dir)
logging.getLogger().removeHandler(h)
assert not any(k in buf.getvalue() for k in ("fallback", "skipping", "identity transform")), buf.getvalue()

carrier = cfg.selected_block[0] - 1
name = f"model.layers.{carrier}.mlp.down_proj.weight"
w_new = read_tensor(cfg.pruned_dir, name).float()
w_base = read_tensor(model_dir(cfg.model_path), name).float()
ratio = (w_new.norm() / w_base.norm()).item()
print(json.load(open(cfg.pruned_dir / "config.json"))["num_hidden_layers"], "layers | block", cfg.selected_block,
      f"| norm ratio {ratio:.4f} (earlier run: 3.0981; above ~3.1 the block did not recover after heal)")

old_pruned = OLD_RUN / "pruned_ReplaceMe_joint_lstsq_1"
if cfg.selected_block == (22, 30) and old_pruned.exists():
    w_old = read_tensor(old_pruned, name).float()
    print(f"vs earlier T^T W: relative difference {((w_new - w_old).norm() / w_old.norm()).item():.2e}, "
          f"max abs {(w_new - w_old).abs().max().item():.2e}")
json.dump({"block_1based": cfg.selected_block, "carrier_0based": carrier, "norm_ratio": round(ratio, 4),
           "lstsq_joint": md5(lj.__file__), "distance_scored": md5(ds.__file__)},
          open(cfg.pruned_dir / "blocks.json", "w"), indent=2)
gc.collect(); torch.cuda.empty_cache()

vram before: 0.02 GiB


2026-10-06 10:21:20 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:21:20 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 10:21:20 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:21:20 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 10:21:24 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:21:25 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"


device map: {'0': 1, '1': 1} | vram GiB: 4.35


...

2026-10-06 10:22:46 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:22:46 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

2026-10-06 10:22:55 [INFO] HTTP Request: GET https://huggingface.co/api/models/unsloth/llama-3-8b/revision/main "HTTP/1.1 200 OK"


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

24 layers | block (22, 30) | norm ratio 3.0989 (earlier run: 3.0981; above ~3.1 the block did not recover after heal)
vs earlier T^T W: relative difference 5.32e-02, max abs 5.86e-02


### 5b. Eval of the pruned model with

In [11]:
if done("pruned_T_only"): print("on disk")
else: run_evaluation(cfg, "pruned_T_only", str(cfg.pruned_dir))
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

2026-10-06 10:23:12 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 10:23:13 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 10:23:13 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 10:23:13 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 10:23:13 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-10-06 10:23:14 [INFO] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=openai/gsm8k "HTTP/1.1 200 OK"
2026-10-06 10:23:14 [I

pruned_T_only: EM 7.43% (98/1319) on 24 layers in 44.6 min -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/results/pruned_T_only_nf4.json


### 6. Heal

In [12]:

if (cfg.heal_adapter_dir / "adapter_config.json").exists(): print("есть:", cfg.heal_adapter_dir)
else:
    vram("before"); train_lora(cfg, str(cfg.pruned_dir), cfg.heal_adapter_dir)
gc.collect(); torch.cuda.empty_cache()

vram before: 0.02 GiB


2026-10-06 11:07:51 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 11:07:51 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 11:07:51 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 11:07:52 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 11:07:52 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-10-06 11:07:52 [INFO] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=openai/gsm8k "HTTP/1.1 200 OK"
2026-10-06 11:07:53 [I

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

trainable params: 10,616,832 || all params: 6,295,982,080 || trainable%: 0.1686


Step,Training Loss
10,1.982238
20,1.573713
30,1.357971
40,1.193898
50,1.162145
60,1.192882
70,1.124830
80,1.101850
90,1.162183
100,1.107729


### 7. Evaluation

In [13]:
if done("pruned_healed"): print("есть")
else: run_evaluation(cfg, "pruned_healed", str(cfg.pruned_dir), str(cfg.heal_adapter_dir))
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

2026-10-06 11:43:49 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 11:43:49 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 11:43:49 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 11:43:50 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 11:43:50 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-10-06 11:43:50 [INFO] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=openai/gsm8k "HTTP/1.1 200 OK"
2026-10-06 11:43:51 [I

pruned_healed: EM 50.11% (661/1319) on 24 layers in 31.4 min -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/results/pruned_healed_nf4.json


### 8. Restore

In [14]:
d = cfg.restored_dir(False)
if not (d / "config.json").exists():
    restore_removed_layers(cfg.model_path, str(cfg.pruned_dir), str(cfg.heal_adapter_dir), str(d), cfg.selected_block, remove_transform=False)
if done("replaceme"): print("есть")
else: run_evaluation(cfg, "replaceme", str(d))
gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

2026-10-06 12:15:29 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 12:15:29 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"
2026-10-06 12:15:29 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 12:15:29 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/config.json "HTTP/1.1 200 OK"


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 12:15:30 [INFO] HTTP Request: HEAD https://huggingface.co/unsloth/llama-3-8b/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
2026-10-06 12:15:30 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/unsloth/llama-3-8b/b0c09b815d8a74b70622c740f108c995c5041a14/generation_config.json "HTTP/1.1 200 OK"


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

2026-10-06 12:16:10 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-10-06 12:16:10 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/openai/gsm8k/740312add88f781978c0658806c59bc2815b9866/README.md "HTTP/1.1 200 OK"
2026-10-06 12:16:10 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 12:16:11 [INFO] HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/openai/gsm8k/openai/gsm8k.py "HTTP/1.1 404 Not Found"
2026-10-06 12:16:11 [INFO] HTTP Request: HEAD https://huggingface.co/datasets/openai/gsm8k/resolve/740312add88f781978c0658806c59bc2815b9866/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-10-06 12:16:11 [INFO] HTTP Request: GET https://datasets-server.huggingface.co/info?dataset=openai/gsm8k "HTTP/1.1 200 OK"
2026-10-06 12:16:11 [I

replaceme: EM 49.73% (656/1319) on 32 layers in 38.7 min -> /home/tnn/LLM_optimisatoin/runs_reproduce/llama-3-8b-repo-nf4/results/replaceme_nf4.json


### 10. Summary

In [26]:
def load(d):
    out = {}
    for f in sorted(Path(d).glob("*.json")):
        r = json.load(open(f)); out[f.stem] = ({int(x["id"]): bool(x["correct"]) for x in r["predictions"]}, r)
    return out
def mcnemar(a, b):
    ids = sorted(set(a) & set(b)); n01 = sum(a[i] and not b[i] for i in ids); n10 = sum(b[i] and not a[i] for i in ids)
    n = n01 + n10; k = min(n01, n10); p = min(1.0, 2 * sum(math.comb(n, j) for j in range(k + 1)) / 2**n) if n else 1.0
    d = (n01 - n10) / len(ids); se = math.sqrt(n - (n01 - n10)**2 / len(ids)) / len(ids) if n else 0.0
    return dict(A_only=n01, B_only=n10, diff_pt=round(d * 100, 2), ci95=f"[{(d - 1.96 * se) * 100:+.2f}; {(d + 1.96 * se) * 100:+.2f}]", p=round(p, 4))

STAGES = [("baseline_nf4", "baseline, 32L"), ("sft_nf4", "fine-tuned, 32L"), ("replaceme_nf4", "restored, 32L")]
LABEL = dict(STAGES)
mine = load(cfg.results_dir)
old = load(OLD_RUN / "results") if (OLD_RUN / "results").exists() else {}
print(f"block {cfg.selected_block}: deleted layers {list(range(*cfg.selected_block))}, T in layer {cfg.selected_block[0] - 1}")
rows = []
for key, label in STAGES:
    if key not in mine: continue
    r = {"model": label, "EM now": round(mine[key][1]["exact_match"] * 100, 2), "correct": mine[key][1]["correct"]}
    if key in old:
        m = mcnemar(mine[key][0], old[key][0])
        r.update({"EM earlier": round(old[key][1]["exact_match"] * 100, 2), "now-earlier, pt": m["diff_pt"],
                  "95% CI": m["ci95"], "p": m["p"]})
    rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

pairs = []
for a, b in [("sft_nf4", "baseline_nf4"), ("sft_nf4", "replaceme_nf4"), ("replaceme_nf4", "baseline_nf4")]:
    if a in mine and b in mine: pairs.append({"A": LABEL[a], "B": LABEL[b], **mcnemar(mine[a][0], mine[b][0])})
pd.DataFrame(pairs)

block (22, 30): deleted layers [22, 23, 24, 25, 26, 27, 28, 29], T in layer 21
          model  EM now  correct  EM earlier  now-earlier, pt         95% CI      p
  baseline, 32L   10.77      142       10.77             0.00 [+0.00; +0.00] 1.0000
fine-tuned, 32L   59.59      786       59.44             0.15 [-1.21; +1.51] 0.9132
  restored, 32L   49.73      656       48.52             1.21 [-0.67; +3.09] 0.2356


,A,B,A_only,B_only,diff_pt,ci95,p
0,"fine-tuned, 32L","baseline, 32L",687,43,48.82,[+45.80; +51.85],0.0
1,"fine-tuned, 32L","restored, 32L",248,118,9.86,[+7.06; +12.65],0.0
2,"restored, 32L","baseline, 32L",568,54,38.97,[+35.92; +42.02],0.0
